<a href="https://colab.research.google.com/github/Binghui5728/SDG6-AI-midterm/blob/main/SDG6_AI_midterm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv('water_potability.csv')
display(df.head())

### 檢查缺失值

In [ ]:
# 檢查每個欄位的缺失值數量
missing_values = df.isnull().sum()
display(missing_values)

# 檢查每個欄位的缺失值百分比
missing_percentage = (df.isnull().sum() / len(df)) * 100
display(missing_percentage.sort_values(ascending=False))

### 處理缺失值

In [ ]:
# 使用每個欄位的平均值來填補缺失值
df['ph'] = df['ph'].fillna(df['ph'].mean())
df['Sulfate'] = df['Sulfate'].fillna(df['Sulfate'].mean())
df['Trihalomethanes'] = df['Trihalomethanes'].fillna(df['Trihalomethanes'].mean())

# 再次檢查缺失值，確認是否已全部填補
print("處理缺失值後的缺失值數量：")
display(df.isnull().sum())

### 異常值檢測 (Outlier Detection)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 選擇數值型特徵進行異常值檢測 (排除目標變數 'Potability')
numerical_cols = df.select_dtypes(include=['float64', 'int64']).columns.tolist()
numerical_cols.remove('Potability')

# 繪製每個數值型特徵的箱形圖
plt.figure(figsize=(15, 10))
for i, col in enumerate(numerical_cols):
    plt.subplot(3, 3, i + 1) # 假設最多9個數值型特徵，根據實際情況調整
    sns.boxplot(y=df[col])
    plt.title(f'{col} Box Plot')
    plt.ylabel('')
plt.tight_layout()
plt.show()

從箱形圖中，我們可以觀察到一些特徵存在異常值。接下來，我們可以根據這些視覺化結果來決定如何處理異常值。常見的方法包括：

*   **移除 (Removal)**: 直接刪除含有異常值的觀測值。
*   **蓋帽 (Capping)**: 將異常值替換為某個上限或下限值 (例如：99百分位數或1百分位數)。
*   **轉換 (Transformation)**: 使用對數轉換等方法來減少異常值的影響。

您希望採用哪種方法來處理異常值呢？或者您想進一步了解某個特定欄位的異常值情況嗎？

### 處理異常值 (蓋帽法 Capping)

In [ ]:
# 再次選擇數值型特徵進行處理 (排除目標變數 'Potability')
numerical_cols = df.select_dtypes(include=['float64', 'int64']).columns.tolist()
numerical_cols.remove('Potability')

# 對每個數值型特徵應用蓋帽法
for col in numerical_cols:
    Q1 = df[col].quantile(0.01) # 1st percentile
    Q99 = df[col].quantile(0.99) # 99th percentile

    # 將低於Q1的值替換為Q1，將高於Q99的值替換為Q99
    df[col] = df[col].clip(lower=Q1, upper=Q99)

print("異常值已使用1%和99%百分位數進行蓋帽處理。")

# 再次繪製箱形圖以檢查處理結果
plt.figure(figsize=(15, 10))
for i, col in enumerate(numerical_cols):
    plt.subplot(3, 3, i + 1)
    sns.boxplot(y=df[col])
    plt.title(f'{col} Box Plot (After Capping)')
    plt.ylabel('')
plt.tight_layout()
plt.show()


### 數據整合與檢查

In [ ]:
# 顯示處理後的資料前五行
display(df.head())

# 顯示資料的資訊 (資料類型、非空值數量)
print("\n處理後資料的資訊：")
display(df.info())

# 顯示資料的描述性統計
print("\n處理後資料的描述性統計：")
display(df.describe())

### 數據轉換 (特徵縮放)

In [ ]:
from sklearn.preprocessing import StandardScaler

# 將特徵 (X) 和目標變數 (y) 分開
X = df.drop('Potability', axis=1)
y = df['Potability']

# 初始化 StandardScaler
scaler = StandardScaler()

# 對特徵數據進行縮放
X_scaled = scaler.fit_transform(X)

# 將縮放後的數據轉換回 DataFrame，並保留原有的欄位名稱
X_scaled_df = pd.DataFrame(X_scaled, columns=X.columns)

print("特徵數據已成功進行標準化處理。")

# 顯示縮放後數據的前五行和描述性統計，以檢查轉換結果
display(X_scaled_df.head())
display(X_scaled_df.describe())

### 數據簡化/歸約 (主成分分析 PCA)

In [ ]:
from sklearn.decomposition import PCA
import numpy as np

# 初始化 PCA 模型
# 我們先不指定 n_components，以便觀察所有主成分的解釋變異量
pca = PCA()

# 對標準化後的數據進行 PCA
X_pca = pca.fit_transform(X_scaled_df)

# 將 PCA 結果轉換回 DataFrame
X_pca_df = pd.DataFrame(X_pca, columns=[f'PC_{i+1}' for i in range(X_pca.shape[1])])

print("數據已進行 PCA 轉換。")

# 顯示每個主成分解釋的變異量比例
print("\n每個主成分解釋的變異量比例：")
display(pca.explained_variance_ratio_)

# 顯示累積解釋變異量
cumulative_variance = np.cumsum(pca.explained_variance_ratio_)
print("\n累積解釋變異量：")
display(cumulative_variance)

# 繪製解釋變異量圖，幫助選擇主成分數量
plt.figure(figsize=(10, 6))
plt.plot(range(1, len(cumulative_variance) + 1), cumulative_variance, marker='o', linestyle='--')
plt.title('解釋變異量與主成分數量')
plt.xlabel('主成分數量')
plt.ylabel('累積解釋變異量')
plt.grid(True)
plt.show()

# 顯示 PCA 轉換後的數據前五行
print("\nPCA 轉換後的數據前五行：")
display(X_pca_df.head())

### 數據劃分 (訓練集與測試集)

In [ ]:
from sklearn.model_selection import train_test_split

# 選擇保留的主成分數量，例如，如果我們希望保留大約 90% 的變異量，從圖中看大約需要 8 個主成分
# 您可以根據之前的累積解釋變異量圖來決定這個數字
n_components_to_retain = 8 # 根據累積解釋變異量圖，保留8個主成分可以達到約 91% 的變異量

# 選擇保留的主成分
X_final = X_pca_df.iloc[:, :n_components_to_retain]

# 將數據集劃分為訓練集和測試集
# test_size=0.2 表示 20% 的數據作為測試集
# random_state 確保每次運行劃分結果一致
X_train, X_test, y_train, y_test = train_test_split(X_final, y, test_size=0.2, random_state=42)

print(f"選擇保留 {n_components_to_retain} 個主成分進行後續分析。")
print(f"訓練集特徵形狀: {X_train.shape}")
print(f"測試集特徵形狀: {X_test.shape}")
print(f"訓練集目標變數形狀: {y_train.shape}")
print(f"測試集目標變數形狀: {y_test.shape}")

display(X_train.head())

### 模型比較：隨機森林 (Random Forest) vs. 支援向量機 (Support Vector Machine)

我們將使用兩種常見的分類模型來預測水質是否可飲用，並比較它們的性能。這兩種模型分別是：

1.  **隨機森林分類器 (Random Forest Classifier)**：這是一種集成學習方法，通過構建多個決策樹並結合它們的預測結果來提高模型的準確性和魯棒性。
2.  **支援向量機分類器 (Support Vector Machine Classifier)**：這是一種強大的二元分類器，通過找到最佳超平面來最大化不同類別數據點之間的分隔邊界。

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("初始化模型並導入評估指標...")

#### 1. 隨機森林分類器 (Random Forest Classifier)

In [ ]:
# 初始化隨機森林分類器
rf_classifier = RandomForestClassifier(random_state=42)

# 訓練模型
print("正在訓練隨機森林模型...")
rf_classifier.fit(X_train, y_train)

# 在測試集上進行預測
y_pred_rf = rf_classifier.predict(X_test)

# 評估模型性能
print("\n隨機森林模型性能評估：")
print(f"準確度 (Accuracy): {accuracy_score(y_test, y_pred_rf):.4f}")
print("\n分類報告 (Classification Report):\n", classification_report(y_test, y_pred_rf))
print("\n混淆矩陣 (Confusion Matrix):\n", confusion_matrix(y_test, y_pred_rf))

#### 2. 支援向量機分類器 (Support Vector Machine Classifier)

In [ ]:
# 初始化支援向量機分類器
# 注意：SVC 對於大規模數據集可能需要較長訓練時間，且參數選擇較為敏感。
# 我們先使用預設參數，並將 `probability=True` 以便之後可以獲取概率預測（如果需要）。
svm_classifier = SVC(random_state=42)

# 訓練模型
print("正在訓練支援向量機模型... (這可能需要一些時間)")
svm_classifier.fit(X_train, y_train)

# 在測試集上進行預測
y_pred_svm = svm_classifier.predict(X_test)

# 評估模型性能
print("\n支援向量機模型性能評估：")
print(f"準確度 (Accuracy): {accuracy_score(y_test, y_pred_svm):.4f}")
print("\n分類報告 (Classification Report):\n", classification_report(y_test, y_pred_svm))
print("\n混淆矩陣 (Confusion Matrix):\n", confusion_matrix(y_test, y_pred_svm))

### 比較結果分析

在兩個模型運行完畢後，您可以根據上述的準確度、分類報告和混淆矩陣來比較它們的性能。通常，您會關注以下幾點：

*   **整體準確度 (Accuracy)**：模型正確預測的樣本比例。
*   **精確度 (Precision)**：模型預測為正類別的樣本中，實際為正類別的比例。
*   **召回率 (Recall)**：實際為正類別的樣本中，模型預測為正類別的比例。
*   **F1-Score**：精確度和召回率的調和平均值，綜合衡量模型的性能。
*   **混淆矩陣**：詳細顯示了真陽性、真陰性、假陽性和假陰性的數量，幫助我們理解模型在哪種類別上表現較好或較差。

根據這些指標，您可以判斷哪種模型在當前的數據集上表現更優。如果需要，我們可以進一步進行超參數調優 (Hyperparameter Tuning) 來優化模型的性能。